# Hybrid CNN + ViT Road Damage Training v2 (Colab T4)

**Fixes over v1:** More epochs (15+25=40), gradient clipping, cosine LR schedule with warmup,
stronger augmentations, 3-stage unfreezing, and proper loss scaling.

**Target: train_loss < 1.0**

In [ ]:
!pip -q install timm==1.0.9 torchmetrics==1.4.0 opencv-python-headless==4.10.0.84

In [ ]:
import json
import math
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from google.colab import drive
drive.mount('/content/drive')

## Clone the Project Repository

This pulls the latest code from your repo into Colab so we can use its utilities.

In [ ]:
import os
if not os.path.exists('/content/Project 2'):
    !git clone https://github.com/LakshmananVS12/projectfy.git "/content/Project 2"
else:
    print('Repo already exists. Pulling latest...')
    !cd "/content/Project 2" && git pull


## Extract and Preprocess Dataset

This extracts the raw zip files from your Google Drive (`/content/drive/MyDrive/rdd2022_raw_zips`) and runs the preprocessing script to prepare them for training.

In [ ]:
import os
import zipfile
import glob

RAW_DRIVE_DIR = '/content/drive/MyDrive/rdd2022_raw_zips'
RAW_EXTRACT_DIR = '/content/Project 2/data/raw/RDD2022'
PROCESSED_DIR = '/content/Project 2/data/processed/rdd2022'

if not os.path.exists(os.path.join(PROCESSED_DIR, 'annotations_train.json')):
    print(f'Processed data not found. Checking for raw zips in {RAW_DRIVE_DIR}...')
    zips = glob.glob(os.path.join(RAW_DRIVE_DIR, '*.zip'))
    if not zips:
        raise FileNotFoundError(f'No zip files found in {RAW_DRIVE_DIR}!')
    
    os.makedirs(RAW_EXTRACT_DIR, exist_ok=True)
    for z in zips:
        print(f'Extracting {z}...')
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall(RAW_EXTRACT_DIR)
    
    print('Extraction complete. Running preprocessing script...')
    !cd "/content/Project 2/ai-service/data" && python preprocess_rdd2022.py --raw-dir "{RAW_EXTRACT_DIR}" --output-dir "{PROCESSED_DIR}" --image-size 640
    print('Preprocessing complete!')
else:
    print('Processed dataset already exists, skipping extraction.')


In [ ]:
PROJECT_ROOT = Path('/content/Project 2')
AI_SERVICE_ROOT = PROJECT_ROOT / 'ai-service'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'rdd2022'
ARTIFACTS_DIR = PROJECT_ROOT / 'training' / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(AI_SERVICE_ROOT) not in sys.path:
    sys.path.append(str(AI_SERVICE_ROOT))

from model.hybrid_detector import HybridDetectorConfig, build_hybrid_model
from model.losses import FCOSLoss
from model.postprocess import decode_detections

# Verify data exists
for f in ['annotations_train.json', 'annotations_val.json']:
    assert (DATA_ROOT / f).exists(), f'Missing {DATA_ROOT / f} -- Please copy your dataset from Drive or run the preprocessing script! See the cells above.'
print('Data verified.')

## Training Configuration

Key changes from v1:
- **3 stages**: head-only warmup (15 ep) -> partial unfreeze (15 ep) -> full fine-tune (10 ep)
- **Gradient clipping** at max_norm=5.0 to prevent exploding gradients in the regression head
- **Cosine annealing** LR schedule with linear warmup for the first 3 epochs of each stage
- **Lower initial LR** (1e-4 for stage 1) since fresh FCOS head needs gentle start
- **Stronger augmentations**: color jitter, random rotation, cutout/erasing

In [ ]:
@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 8
    # Stage 1: Train ONLY fusion + detection head (backbones frozen)
    epochs_stage1: int = 30
    lr_stage1: float = 1e-4        # gentle start for fresh head
    # Stage 2: Unfreeze last ViT blocks + CNN last stage
    epochs_stage2: int = 120
    lr_stage2: float = 3e-5        # lower for pretrained layers
    # Stage 3: Full fine-tune with very low LR
    epochs_stage3: int = 100
    lr_stage3: float = 1e-5
    # Shared
    weight_decay: float = 1e-4
    grad_clip_norm: float = 5.0    # NEW: gradient clipping
    warmup_epochs: int = 3         # NEW: linear warmup per stage
    num_workers: int = 2
    score_threshold_eval: float = 0.20  # slightly lower for early training eval
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: int = 42

cfg = TrainConfig()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if cfg.device == 'cuda':
    torch.backends.cudnn.benchmark = True

print(cfg)
print(f'Total epochs: {cfg.epochs_stage1 + cfg.epochs_stage2 + cfg.epochs_stage3}')

## Dataset with Stronger Augmentations

In [ ]:
class CocoDamageDataset(Dataset):
    def __init__(self, images_dir: Path, annotations_file: Path, train: bool):
        self.images_dir = images_dir
        self.train = train

        data = json.loads(annotations_file.read_text(encoding='utf-8'))
        self.images = data['images']
        self.categories = data['categories']

        by_image = {}
        for ann in data['annotations']:
            by_image.setdefault(ann['image_id'], []).append(ann)
        self.by_image = by_image

    def __len__(self):
        return len(self.images)

    def _augment(self, image: np.ndarray, boxes: torch.Tensor):
        if not self.train:
            return image, boxes

        h, w = image.shape[:2]

        # Horizontal flip (50%)
        if random.random() < 0.5:
            image = image[:, ::-1, :].copy()
            if boxes.shape[0] > 0:
                x1 = boxes[:, 0].clone()
                x2 = boxes[:, 2].clone()
                boxes[:, 0] = w - x2
                boxes[:, 2] = w - x1

        # Color jitter (brightness, contrast, saturation)
        if random.random() < 0.5:
            # Brightness
            delta = random.uniform(-30, 30)
            image = np.clip(image.astype(np.float32) + delta, 0, 255).astype(np.uint8)

        if random.random() < 0.4:
            # Contrast
            factor = random.uniform(0.7, 1.3)
            mean = image.mean()
            image = np.clip((image.astype(np.float32) - mean) * factor + mean, 0, 255).astype(np.uint8)

        if random.random() < 0.3:
            # Convert to HSV and jitter saturation
            hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 1] *= random.uniform(0.7, 1.3)
            hsv = np.clip(hsv, 0, 255).astype(np.uint8)
            image = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)

        # Random erasing / cutout (30%) - helps generalization
        if random.random() < 0.3:
            eh = random.randint(h // 10, h // 4)
            ew = random.randint(w // 10, w // 4)
            ey = random.randint(0, h - eh)
            ex = random.randint(0, w - ew)
            image[ey:ey+eh, ex:ex+ew] = np.random.randint(0, 255, (eh, ew, 3), dtype=np.uint8)

        # Gaussian blur (20%)
        if random.random() < 0.2:
            ksize = random.choice([3, 5])
            image = cv2.GaussianBlur(image, (ksize, ksize), 0)

        return image, boxes

    def __getitem__(self, idx):
        item = self.images[idx]
        image = cv2.imread(str(self.images_dir / item['file_name']))
        if image is None:
            # Fallback: return a blank image if file is corrupt
            image = np.zeros((640, 640, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        anns = self.by_image.get(item['id'], [])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            if w < 2 or h < 2:  # skip degenerate boxes
                continue
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.float32)

        image, boxes = self._augment(image, boxes)

        image = np.ascontiguousarray(image)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if boxes.shape[0] == 0:
            target = torch.zeros((0, 5), dtype=torch.float32)
        else:
            target = torch.cat([boxes, labels.unsqueeze(1)], dim=1)

        return image_tensor, target


def collate_fn(batch):
    images = torch.stack([x[0] for x in batch], dim=0)
    targets = [x[1] for x in batch]
    return images, targets

In [ ]:
train_ds = CocoDamageDataset(
    images_dir=DATA_ROOT / 'images' / 'train',
    annotations_file=DATA_ROOT / 'annotations_train.json',
    train=True,
)
val_ds = CocoDamageDataset(
    images_dir=DATA_ROOT / 'images' / 'val',
    annotations_file=DATA_ROOT / 'annotations_val.json',
    train=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
    drop_last=True,  # avoid tiny last batch that destabilizes BN
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

print(f'Train images: {len(train_ds)}, Val images: {len(val_ds)}')
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## Model, Loss, Scheduler

In [ ]:
model_cfg = HybridDetectorConfig(
    num_classes=4,
    class_names=('pothole', 'linear_crack', 'alligator_crack', 'edge_break'),
    cnn_variant='resnet34',
    cnn_output_stage='layer3',
    vit_variant='deit_small_patch16_224',
    pretrained_backbones=True,
)

model = build_hybrid_model(model_cfg).to(cfg.device)
criterion = FCOSLoss(num_classes=model_cfg.num_classes, stride=model.stride)
scaler = torch.amp.GradScaler(enabled=(cfg.device == 'cuda'))

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')


def build_optimizer(lr: float):
    trainable = [p for p in model.parameters() if p.requires_grad]
    print(f'  Trainable parameters: {sum(p.numel() for p in trainable):,}')
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=cfg.weight_decay)


def build_scheduler(optimizer, num_epochs, warmup_epochs):
    """Cosine annealing with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs  # linear warmup
        progress = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine decay
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

## Evaluation Utilities

In [ ]:
def to_metric_format(decoded, targets):
    preds = []
    refs = []

    for pred_det, tgt in zip(decoded, targets):
        if len(pred_det) == 0:
            preds.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'scores': torch.zeros((0,), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            boxes = torch.tensor([d['bbox'] for d in pred_det], dtype=torch.float32)
            scores = torch.tensor([d['confidence'] for d in pred_det], dtype=torch.float32)
            labels = torch.tensor([d['class_id'] for d in pred_det], dtype=torch.int64)
            preds.append({'boxes': boxes, 'scores': scores, 'labels': labels})

        if tgt.shape[0] == 0:
            refs.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            refs.append({
                'boxes': tgt[:, :4].cpu().float(),
                'labels': tgt[:, 4].cpu().long(),
            })

    return preds, refs


def iou_xyxy(a: torch.Tensor, b: torch.Tensor) -> float:
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def batch_iou_f1(decoded, targets, iou_thresh=0.5):
    tp = 0
    fp = 0
    fn = 0
    iou_sum = 0.0
    match_count = 0

    for pred_det, tgt in zip(decoded, targets):
        gt = tgt.cpu()
        used_gt = set()

        for pred in pred_det:
            pbox = torch.tensor(pred['bbox'], dtype=torch.float32)
            plabel = int(pred['class_id'])

            best_iou = 0.0
            best_idx = -1
            for gi in range(gt.shape[0]):
                if gi in used_gt:
                    continue
                if int(gt[gi, 4].item()) != plabel:
                    continue
                iou = iou_xyxy(pbox, gt[gi, :4])
                if iou > best_iou:
                    best_iou = iou
                    best_idx = gi

            if best_idx >= 0 and best_iou >= iou_thresh:
                tp += 1
                used_gt.add(best_idx)
                iou_sum += best_iou
                match_count += 1
            else:
                fp += 1

        fn += max(0, gt.shape[0] - len(used_gt))

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)
    mean_iou = iou_sum / (match_count + 1e-8)
    return float(mean_iou), float(f1)


@torch.no_grad()
def run_eval(model, loader):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    iou_vals = []
    f1_vals = []

    model.eval()
    for images, targets in loader:
        images = images.to(cfg.device, non_blocking=True)
        with torch.autocast(device_type='cuda', enabled=(cfg.device == 'cuda'), dtype=torch.float16):
            outputs = model(images)

        decoded = decode_detections(
            cls_logits=outputs['cls_logits'].float(),
            bbox_reg=outputs['bbox_reg'].float(),
            centerness=outputs['centerness'].float(),
            stride=model.stride,
            class_names=model_cfg.class_names,
            score_threshold=cfg.score_threshold_eval,
            nms_iou_threshold=0.5,
            top_k=200,
            image_hw=(cfg.image_size, cfg.image_size),
        )

        preds, refs = to_metric_format(decoded, targets)
        metric.update(preds, refs)

        mean_iou, f1 = batch_iou_f1(decoded, targets, iou_thresh=0.5)
        iou_vals.append(mean_iou)
        f1_vals.append(f1)

    map_scores = metric.compute()
    return {
        'mAP': float(map_scores.get('map', torch.tensor(0.0)).item()),
        'mAP@50': float(map_scores.get('map_50', torch.tensor(0.0)).item()),
        'mean_iou': float(np.mean(iou_vals) if iou_vals else 0.0),
        'f1': float(np.mean(f1_vals) if f1_vals else 0.0),
    }

## Training Loop with Gradient Clipping + Cosine Schedule

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, grad_clip_norm):
    model.train()
    running_loss = 0.0
    running_cls = 0.0
    running_reg = 0.0
    running_ctr = 0.0
    num_batches = 0

    for images, targets in loader:
        images = images.to(cfg.device, non_blocking=True)
        targets = [t.to(cfg.device) for t in targets]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', enabled=(cfg.device == 'cuda'), dtype=torch.float16):
            outputs = model(images)
            losses = criterion(outputs['cls_logits'], outputs['bbox_reg'], outputs['centerness'], targets)
            loss = losses['loss_total']

        scaler.scale(loss).backward()

        # CRITICAL: Gradient clipping to prevent exploding gradients in regression head
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)

        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.item())
        running_cls += float(losses['loss_cls'].item())
        running_reg += float(losses['loss_reg'].item())
        running_ctr += float(losses['loss_center'].item())
        num_batches += 1

    n = max(1, num_batches)
    return {
        'total': running_loss / n,
        'cls': running_cls / n,
        'reg': running_reg / n,
        'ctr': running_ctr / n,
    }

In [ ]:
def run_stage(stage_num, num_epochs, lr, warmup_epochs):
    optimizer = build_optimizer(lr)
    scheduler = build_scheduler(optimizer, num_epochs, warmup_epochs)

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        loss_dict = train_one_epoch(model, train_loader, optimizer, criterion, scaler, cfg.grad_clip_norm)
        scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        # Evaluate every 3 epochs to save time, and always on last epoch
        metrics = {}
        if epoch % 3 == 0 or epoch == num_epochs:
            metrics = run_eval(model, val_loader)

        row = {
            'stage': stage_num,
            'epoch': epoch,
            'train_loss': round(loss_dict['total'], 4),
            'loss_cls': round(loss_dict['cls'], 4),
            'loss_reg': round(loss_dict['reg'], 4),
            'loss_ctr': round(loss_dict['ctr'], 4),
            'lr': current_lr,
            'time_s': round(elapsed, 1),
            **metrics,
        }
        history.append(row)

        # Print progress
        metric_str = ''
        if metrics:
            metric_str = f" | mAP50={metrics.get('mAP@50', 0):.4f} F1={metrics.get('f1', 0):.4f}"
        print(f"  S{stage_num} E{epoch:02d} | loss={loss_dict['total']:.4f} "
              f"(cls={loss_dict['cls']:.4f} reg={loss_dict['reg']:.4f} ctr={loss_dict['ctr']:.4f}) "
              f"| lr={current_lr:.2e} | {elapsed:.0f}s{metric_str}")

        # Save best checkpoint based on mAP@50 (when available) or lowest loss
        global best_map50, best_loss
        save_reason = None
        if metrics and metrics.get('mAP@50', 0) > best_map50:
            best_map50 = metrics['mAP@50']
            save_reason = f'best mAP@50={best_map50:.4f}'
        elif loss_dict['total'] < best_loss:
            best_loss = loss_dict['total']
            save_reason = f'best loss={best_loss:.4f}'

        if save_reason:
            torch.save({
                'model_state': model.state_dict(),
                'config': model_cfg.__dict__,
                'metrics': row,
            }, best_ckpt)
            print(f'    >> Saved checkpoint ({save_reason})')


# ---- Initialize tracking ----
history = []
best_map50 = -1.0
best_loss = float('inf')
best_ckpt = ARTIFACTS_DIR / 'hybrid_detector_best.pt'

# ================ STAGE 1: Head-only warmup ================
print('\n=== STAGE 1: Training fusion + detection head (backbones frozen) ===')
model.freeze_backbones()
run_stage(1, cfg.epochs_stage1, cfg.lr_stage1, cfg.warmup_epochs)

# ================ STAGE 2: Partial unfreeze ================
print('\n=== STAGE 2: Unfreezing last ViT blocks + CNN last stage ===')
model.unfreeze_vit_last_blocks(num_blocks=2)
model.unfreeze_cnn_last_stage()
run_stage(2, cfg.epochs_stage2, cfg.lr_stage2, cfg.warmup_epochs)

# ================ STAGE 3: Full fine-tune ================
print('\n=== STAGE 3: Full fine-tune (all layers) ===')
for param in model.parameters():
    param.requires_grad = True
run_stage(3, cfg.epochs_stage3, cfg.lr_stage3, 2)  # shorter warmup

# ---- Save training history ----
history_path = ARTIFACTS_DIR / 'train_history.json'
history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
print(f'\nTraining complete! Best mAP@50: {best_map50:.4f}, Best loss: {best_loss:.4f}')
print(f'Best checkpoint: {best_ckpt}')

## ONNX Export

In [ ]:
# Load best weights before ONNX export
state = torch.load(ARTIFACTS_DIR / 'hybrid_detector_best.pt', map_location=cfg.device)
model.load_state_dict(state['model_state'])
model.eval()
print('Best checkpoint metrics:', state.get('metrics', {}))

class OnnxExportWrapper(nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector

    def forward(self, x):
        out = self.detector(x)
        return out['cls_logits'], out['bbox_reg'], out['centerness']

wrapper = OnnxExportWrapper(model).to(cfg.device)
dummy = torch.randn(1, 3, cfg.image_size, cfg.image_size, device=cfg.device)
onnx_path = ARTIFACTS_DIR / 'hybrid_detector.onnx'

torch.onnx.export(
    wrapper,
    dummy,
    str(onnx_path),
    input_names=['image'],
    output_names=['cls_logits', 'bbox_reg', 'centerness'],
    dynamic_axes={
        'image': {0: 'batch'},
        'cls_logits': {0: 'batch'},
        'bbox_reg': {0: 'batch'},
        'centerness': {0: 'batch'},
    },
    opset_version=17,
)

print('ONNX exported:', onnx_path)

## Copy Artifacts to Drive & Project Weights Folder

In [ ]:
# Copy to Google Drive for easy download
DRIVE_OUT = Path('/content/drive/MyDrive/road-damage-artifacts')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# Also copy to project weights dir (if running from cloned repo)
WEIGHTS_DIR = AI_SERVICE_ROOT / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

for src in [
    ARTIFACTS_DIR / 'hybrid_detector_best.pt',
    ARTIFACTS_DIR / 'hybrid_detector.onnx',
    ARTIFACTS_DIR / 'train_history.json',
]:
    if src.exists():
        for dest_dir in [DRIVE_OUT, WEIGHTS_DIR]:
            dst = dest_dir / src.name
            dst.write_bytes(src.read_bytes())
            print(f'Copied: {src.name} -> {dst}')

print('\nDone! Download hybrid_detector_best.pt and hybrid_detector.onnx')
print('Place them in: d:/Project 2/ai-service/weights/')

## Summary of Changes from v1

| Issue in v1 | Fix in v2 |
|---|---|
| Only 10 epochs total | 40 epochs (15+15+10) across 3 stages |
| No gradient clipping | clip_grad_norm at 5.0 |
| Flat learning rate | Cosine annealing + 3-epoch linear warmup |
| LR too high (3e-4 for fresh head) | Stage 1: 1e-4, Stage 2: 3e-5, Stage 3: 1e-5 |
| Weak augmentation (flip + brightness) | Added contrast, saturation, cutout, Gaussian blur |
| 2-stage training | 3-stage: head-only -> partial -> full fine-tune |
| No loss breakdown | Separate cls/reg/centerness loss logging |
| Eval every epoch (slow) | Eval every 3 epochs + last epoch |

## Expected Outputs

After successful run, confirm these files exist:
- `training/artifacts/hybrid_detector_best.pt`
- `training/artifacts/hybrid_detector.onnx`
- `training/artifacts/train_history.json`

Copy the weight and ONNX files back into this repo:
- `ai-service/weights/hybrid_detector_best.pt`
- `ai-service/weights/hybrid_detector.onnx`